# Gold-Set Annotation Tool

Hand-label 75 stratified SPY H1 candles for FVG ground truth. Click a button per candle. Progress saves to `data/gold_labels.csv` after each click. Close the tab any time, reopen to resume.

**How to read the chart:** the red dashed line marks the *target candle* (annotated `N`). The candle right before it (annotated `N-1`) is the middle of the 3-candle FVG pattern. The candle before *that* — 2 bars before the red line — is the first bar of the FVG pattern.

**Decide:** is there a Fair Value Gap pattern in the 3 bars *up to and including the red line*?

- **Bullish FVG (buy bias zone)** — price gapped up. The candle 2 bars before the red line has its high *below* the low of the red-line candle (i.e. `high[N-2] < low[N]`). The middle bar (N-1) is a green body.
- **Bearish FVG (sell bias zone)** — price gapped down. The candle 2 bars before the red line has its low *above* the high of the red-line candle (i.e. `low[N-2] > high[N]`). The middle bar (N-1) is a red body.
- **None** — no clear gap, or pattern doesn't qualify.
- **Ambiguous** — could be either way; saved as `none` with a note.

**The cell below pre-selects the rule's suggestion for each candle.** Click `Confirm` to accept or click an alternative button to override. This way you only think hard when you disagree with the rule.

Run the cells in order. The third cell renders the UI.

In [1]:
# Setup: load data + gold set + previous annotations
import sys, csv
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

from src.data.annotate import sample_gold_set, compute_kappa
from src.data.labels.sr import compute_pivot_levels
from src.data.labels.valid_fvg import ValidFVGLabeller

DATA_PATH = ROOT / 'data' / 'processed' / 'spy_h1.parquet'
OUTPUT_PATH = ROOT / 'data' / 'gold_labels.csv'

df = pd.read_parquet(DATA_PATH)
n_df = len(df)
print(f'Loaded {n_df} H1 bars')

# Pre-compute overlay data once
pivot_high_full, pivot_low_full = compute_pivot_levels(df, lookback=5)
swing_high_full = df["high"].rolling(50, min_periods=1).max().shift(1)   # PRIOR 50-bar swing high
swing_low_full  = df["low"].rolling(50, min_periods=1).min().shift(1)    # PRIOR 50-bar swing low
swing_mid_full  = swing_low_full + 0.5 * (swing_high_full - swing_low_full)

# Per-bar BOS occurrence — matches labeller math.
# close > prior 50-bar swing high  → bull BOS at that bar
# close < prior 50-bar swing low   → bear BOS at that bar
bull_bos_per_bar = (df["close"] > swing_high_full).fillna(False)
bear_bos_per_bar = (df["close"] < swing_low_full).fillna(False)
print('Overlay data pre-computed (S/R, swing, Gann mid, per-bar BOS).')

# Sample gold set using ValidFVGLabeller (N+2 convention)
labeller = ValidFVGLabeller()
labels_series = labeller.label(df[["open", "high", "low", "close", "volume"]])
df_labelled = df.copy()
df_labelled["raw_label"] = labels_series

gold_set = sample_gold_set(df_labelled)
print(f'Gold set: {len(gold_set)} candles')

annotated_indices = set()
if OUTPUT_PATH.exists() and OUTPUT_PATH.stat().st_size > 0:
    try:
        prior = pd.read_csv(OUTPUT_PATH)
        if 'candle_index' in prior.columns:
            annotated_indices = set(prior['candle_index'].tolist())
            print(f'Resuming: {len(annotated_indices)}/{len(gold_set)} already annotated')
        else:
            print('CSV has no candle_index column — treating as empty')
    except pd.errors.EmptyDataError:
        print('CSV exists but is empty — starting fresh')
else:
    print('No prior annotations — starting fresh')

remaining = gold_set[~gold_set['candle_index'].isin(annotated_indices)].reset_index(drop=True)
print(f'Remaining: {len(remaining)} candles to label')
print()
print('See docs/fvg-label-guide.md for the 5-step decision flow.')


Loaded 14686 H1 bars
Overlay data pre-computed (S/R, swing, Gann mid, per-bar BOS).
Gold set: 75 candles
Resuming: 70/75 already annotated
Remaining: 5 candles to label

See docs/fvg-label-guide.md for the 5-step decision flow.


In [2]:
# Annotation UI — chart window CENTERED on the FVG middle candle (label_pos - 2).
# NEW: explicit BOS markers — yellow stars on candles that triggered a rule-detected BOS.

import html as _html_escape

FIELDNAMES = ['candle_index', 'datetime', 'programmatic_label', 'human_label', 'annotator_note']
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

if not OUTPUT_PATH.exists() or OUTPUT_PATH.stat().st_size == 0:
    with open(OUTPUT_PATH, 'w', newline='') as f:
        csv.DictWriter(f, fieldnames=FIELDNAMES).writeheader()

state = {'idx': 0, 'remaining': remaining.copy()}

GREEN = '#1f9d55'
RED   = '#cc1f1a'
DOJI  = '#888888'
BODY_HALF_WIDTH = 0.40
BOS_BULL_COLOR = '#f5a623'   # orange-gold star for bullish BOS
BOS_BEAR_COLOR = '#9013fe'   # purple-pink star for bearish BOS

progress_label   = widgets.HTML()
suggestion_label = widgets.HTML()
status_label     = widgets.HTML()
chart_html       = widgets.HTML(value='', layout=widgets.Layout(height='760px'))

btn_bull = widgets.Button(description='b — Bull FVG', button_style='success', layout=widgets.Layout(width='160px', height='52px'))
btn_bear = widgets.Button(description='e — Bear FVG', button_style='danger',  layout=widgets.Layout(width='160px', height='52px'))
btn_none = widgets.Button(description='n — None',     button_style='',        layout=widgets.Layout(width='160px', height='52px'))
btn_amb  = widgets.Button(description='a — Ambiguous',button_style='warning', layout=widgets.Layout(width='160px', height='52px'))

button_row = widgets.HBox([btn_bull, btn_bear, btn_none, btn_amb])


def _fvg_zone_at(middle_idx: int):
    if middle_idx < 1 or middle_idx > n_df - 2:
        return None
    h_prev = float(df["high"].iloc[middle_idx - 1])
    l_next = float(df["low"].iloc[middle_idx + 1])
    l_prev = float(df["low"].iloc[middle_idx - 1])
    h_next = float(df["high"].iloc[middle_idx + 1])
    o_mid  = float(df["open"].iloc[middle_idx])
    c_mid  = float(df["close"].iloc[middle_idx])
    if h_prev < l_next and c_mid > o_mid:
        return ("bull", h_prev, l_next)
    if l_prev > h_next and c_mid < o_mid:
        return ("bear", h_next, l_prev)
    return None


def _build_figure(candle_idx: int) -> go.Figure:
    middle = candle_idx - 2
    start = max(0, middle - 50)  # widened to cover full 50-bar BOS lookback
    end   = min(n_df, middle + 6)
    window_df  = df.iloc[start:end]
    n_win      = len(window_df)
    local_idx  = middle - start

    x_pos  = list(range(n_win))
    opens  = window_df['open'].tolist()
    highs  = window_df['high'].tolist()
    lows   = window_df['low'].tolist()
    closes = window_df['close'].tolist()

    wick_x, wick_y = [], []
    for i in x_pos:
        wick_x += [i, i, None]
        wick_y += [lows[i], highs[i], None]

    tick_step    = max(1, n_win // 6)
    tick_indices = list(range(0, n_win, tick_step))
    if (n_win - 1) not in tick_indices:
        tick_indices.append(n_win - 1)
    tick_text = [window_df.index[i].strftime('%m-%d %H:%M') for i in tick_indices]

    swing_mid_at = float(swing_mid_full.iloc[middle]) if not np.isnan(swing_mid_full.iloc[middle]) else None
    ph_win = pivot_high_full.iloc[start:end]
    pl_win = pivot_low_full.iloc[start:end]

    ph_x, ph_y, pl_x, pl_y = [], [], [], []
    for j in x_pos:
        ph_val = ph_win.iloc[j]
        pl_val = pl_win.iloc[j]
        if not np.isnan(ph_val) and abs(highs[j] - ph_val) < 0.05:
            ph_x.append(j); ph_y.append(highs[j] + 0.20)
        if not np.isnan(pl_val) and abs(lows[j] - pl_val) < 0.05:
            pl_x.append(j); pl_y.append(lows[j] - 0.20)

    # BOS markers — ONLY for bars LEFT of N (causally relevant for Q4)
    # within last 50 bars before N (matches labeller's bos_lookback=50)
    bos_window_start = max(start, middle - 50)
    bull_bos_x, bull_bos_y = [], []
    bear_bos_x, bear_bos_y = [], []
    bos_level_lines = []  # store (prior_swing_level, x_at_break) tuples for drawing
    for j in x_pos:
        global_j = start + j
        if global_j >= middle:
            break  # only bars strictly before N count
        if global_j < bos_window_start:
            continue
        if bool(bull_bos_per_bar.iloc[global_j]):
            bull_bos_x.append(j)
            bull_bos_y.append(highs[j] + 0.40)
            bos_level_lines.append(('bull', j, float(swing_high_full.iloc[global_j])))
        if bool(bear_bos_per_bar.iloc[global_j]):
            bear_bos_x.append(j)
            bear_bos_y.append(lows[j] - 0.40)
            bos_level_lines.append(('bear', j, float(swing_low_full.iloc[global_j])))

    fvg = _fvg_zone_at(middle)
    shapes = []

    if fvg is not None:
        direction, zone_bot, zone_top = fvg
        if direction == 'bull':
            fill, line_c = 'rgba(31,157,85,0.20)', 'rgba(31,157,85,0.95)'
        else:
            fill, line_c = 'rgba(204,31,26,0.20)', 'rgba(204,31,26,0.95)'
        shapes.append(dict(
            type='rect', xref='x', yref='y',
            x0=-0.5, x1=n_win - 0.5, y0=zone_bot, y1=zone_top,
            fillcolor=fill, layer='below', line=dict(color=line_c, width=2),
        ))

    for i in x_pos:
        o = opens[i]; c = closes[i]
        if c > o:   color = GREEN
        elif c < o: color = RED
        else:       color = DOJI
        body_top = max(o, c); body_bot = min(o, c)
        if body_top - body_bot < 0.03:
            mid_p = 0.5 * (body_top + body_bot)
            body_top = mid_p + 0.015; body_bot = mid_p - 0.015
        shapes.append(dict(
            type='rect', xref='x', yref='y',
            x0=i - BODY_HALF_WIDTH, x1=i + BODY_HALF_WIDTH,
            y0=body_bot, y1=body_top,
            fillcolor=color, line=dict(color=color, width=1), layer='above',
        ))

    # BOS broken-level horizontal segments — draw the level that was broken,
    # from leftmost visible bar up to the breaking bar
    for direction, x_break, level in bos_level_lines:
        col = BOS_BULL_COLOR if direction == 'bull' else BOS_BEAR_COLOR
        shapes.append(dict(
            type='line', xref='x', yref='y',
            x0=-0.5, x1=x_break,
            y0=level, y1=level,
            line=dict(color=col, dash='dashdot', width=1.4),
            layer='above',
        ))

    # N target — red dashed
    shapes.append(dict(type='line', xref='x', yref='paper',
        x0=local_idx, x1=local_idx, y0=0, y1=1,
        line=dict(color='#d0021b', dash='dash', width=2.5), layer='above'))

    # N+2 reaction — green dotted
    if local_idx + 2 < n_win:
        shapes.append(dict(type='line', xref='x', yref='paper',
            x0=local_idx + 2, x1=local_idx + 2, y0=0, y1=1,
            line=dict(color='#1f9d55', dash='dot', width=2), layer='above'))

    if swing_mid_at is not None:
        shapes.append(dict(type='line', xref='x', yref='y',
            x0=-0.5, x1=n_win - 0.5, y0=swing_mid_at, y1=swing_mid_at,
            line=dict(color='#1f6dd0', dash='dash', width=1.5)))

    candidate_levels = []
    seen = set()
    for v in pd.concat([ph_win, pl_win]).dropna().tolist():
        v = round(float(v), 4)
        if v not in seen:
            candidate_levels.append(v); seen.add(v)
    zone_centre = (0.5 * (fvg[1] + fvg[2])) if fvg else float(closes[local_idx])
    candidate_levels.sort(key=lambda v: abs(v - zone_centre))
    for v in candidate_levels[:2]:
        shapes.append(dict(type='line', xref='x', yref='y',
            x0=-0.5, x1=n_win - 0.5, y0=v, y1=v,
            line=dict(color='#ff8c00', dash='dot', width=1.2)))

    annotations = []
    for offset, label_text, color in [(0, 'N (target)', '#d0021b'), (2, 'N+2 (reaction)', '#1f9d55')]:
        pos = local_idx + offset
        if 0 <= pos < n_win:
            annotations.append(dict(
                x=pos, y=highs[pos],
                text=f"<b>{label_text}</b>", showarrow=False, yshift=24,
                font=dict(size=12, color=color),
                bgcolor='rgba(255,255,255,0.95)',
                bordercolor=color, borderwidth=1, borderpad=3))

    title_txt = (
        f"Label idx {candle_idx} → FVG middle N={middle} &nbsp;|&nbsp; "
        f"N at {df.index[middle].strftime('%Y-%m-%d %H:%M')} &nbsp;|&nbsp; "
        + (f"<b style='color:{(GREEN if fvg[0]=='bull' else RED)}'>"
           f"{fvg[0].upper()} FVG zone visible</b>" if fvg else "<i>no FVG geometry here</i>")
    )

    y_lo = min(lows); y_hi = max(highs)
    if fvg is not None:
        y_lo = min(y_lo, fvg[1]); y_hi = max(y_hi, fvg[2])
    # Extend Y to fit any BOS broken levels too
    for _, _, lvl in bos_level_lines:
        y_lo = min(y_lo, lvl); y_hi = max(y_hi, lvl)
    y_pad = max((y_hi - y_lo) * 0.10, 0.5)

    fig = go.Figure(data=[
        go.Scatter(x=wick_x, y=wick_y, mode='lines',
                   line=dict(color='#222', width=1.4), hoverinfo='skip', showlegend=False),
        go.Scatter(x=ph_x, y=ph_y, mode='markers',
                   marker=dict(symbol='triangle-down', color='#7a3da6', size=12,
                               line=dict(color='white', width=1)),
                   hoverinfo='skip', showlegend=False),
        go.Scatter(x=pl_x, y=pl_y, mode='markers',
                   marker=dict(symbol='triangle-up', color='#7a3da6', size=12,
                               line=dict(color='white', width=1)),
                   hoverinfo='skip', showlegend=False),
        go.Scatter(x=bull_bos_x, y=bull_bos_y, mode='markers+text',
                   marker=dict(symbol='star', color=BOS_BULL_COLOR, size=16,
                               line=dict(color='black', width=1)),
                   text=['↑BOS'] * len(bull_bos_x), textposition='top center',
                   textfont=dict(size=9, color=BOS_BULL_COLOR),
                   hoverinfo='skip', showlegend=False),
        go.Scatter(x=bear_bos_x, y=bear_bos_y, mode='markers+text',
                   marker=dict(symbol='star', color=BOS_BEAR_COLOR, size=16,
                               line=dict(color='black', width=1)),
                   text=['↓BOS'] * len(bear_bos_x), textposition='bottom center',
                   textfont=dict(size=9, color=BOS_BEAR_COLOR),
                   hoverinfo='skip', showlegend=False),
    ])
    fig.update_layout(
        title=title_txt,
        shapes=shapes, annotations=annotations,
        xaxis_rangeslider_visible=False, height=720,
        margin=dict(l=60, r=60, t=70, b=70),
        showlegend=False, plot_bgcolor='#fafafa', paper_bgcolor='white',
        font=dict(family='Helvetica, Arial', size=12),
        xaxis=dict(tickmode='array', tickvals=tick_indices, ticktext=tick_text,
                   tickangle=0, range=[-0.5, n_win - 0.5], showgrid=False),
        yaxis=dict(showgrid=True, gridcolor='#ececec', zeroline=False, side='right',
                   range=[y_lo - y_pad, y_hi + y_pad]),
    )
    return fig


def _fig_to_iframe(fig: go.Figure) -> str:
    inner = fig.to_html(
        include_plotlyjs='cdn', full_html=True,
        config={'displayModeBar': False},
    )
    escaped = _html_escape.escape(inner, quote=True)
    return (
        f'<iframe srcdoc="{escaped}" '
        f'style="width:100%; height:640px; border:0;" '
        f'sandbox="allow-scripts allow-same-origin"></iframe>'
    )


def render_current() -> None:
    rem = state['remaining']
    done = len(gold_set) - len(rem) + state['idx']
    progress_label.value = f"<h3 style='margin:6px 0'>Progress: {done}/{len(gold_set)} candles labelled</h3>"

    if state['idx'] >= len(rem):
        suggestion_label.value = (
            "<h3 style='color:green;'>All candles annotated. "
            "Run the kappa cell below to measure agreement.</h3>"
        )
        for b in (btn_bull, btn_bear, btn_none, btn_amb):
            b.disabled = True
        chart_html.value = ''
        return

    row = rem.iloc[state['idx']]
    candle_idx = int(row['candle_index'])
    suggestion_label.value = (
        f"<div style='padding:10px;background:#f0f7ff;border-left:4px solid #4a90e2;font-size:14px;'>"
        f"<b>Label idx {candle_idx}</b> (FVG middle N={candle_idx-2}) &nbsp;|&nbsp; "
        f"Programmatic guess hidden. "
        f"See <code>docs/fvg-label-guide.md</code> for the decision flow."
        f"</div>"
    )

    fig = _build_figure(candle_idx)
    chart_html.value = _fig_to_iframe(fig)


def save_and_advance(human_label: int, note: str) -> None:
    rem = state['remaining']
    if state['idx'] >= len(rem):
        return
    row = rem.iloc[state['idx']]
    candle_idx = int(row['candle_index'])
    with open(OUTPUT_PATH, 'a', newline='') as f:
        csv.DictWriter(f, fieldnames=FIELDNAMES).writerow({
            'candle_index': candle_idx,
            'datetime': str(df.index[candle_idx]),
            'programmatic_label': int(row['programmatic_label']),
            'human_label': human_label,
            'annotator_note': note,
        })
    saved_name = {1: 'Bull FVG', -1: 'Bear FVG', 0: 'None'}.get(human_label, '?')
    status_label.value = (
        f"<div style='padding:6px;background:#eef9ee;border-left:4px solid #2a8c2a;'>"
        f"Saved candle {candle_idx} → <b>{saved_name}</b>{(' (note: '+note+')') if note else ''}"
        f"</div>"
    )
    state['idx'] += 1
    render_current()


btn_bull.on_click(lambda b: save_and_advance(1, ''))
btn_bear.on_click(lambda b: save_and_advance(-1, ''))
btn_none.on_click(lambda b: save_and_advance(0, ''))
btn_amb.on_click( lambda b: save_and_advance(0, 'ambiguous'))

display(progress_label, suggestion_label, chart_html, button_row, status_label)
render_current()


HTML(value='')

HTML(value='')

HTML(value='', layout=Layout(height='760px'))

HTML(value='')

In [5]:
# After all 75 candles annotated — compute Cohen kappa and check the gate
import math
import pandas as pd

if not OUTPUT_PATH.exists() or OUTPUT_PATH.stat().st_size == 0:
    print('No annotations yet — go back to the UI cell.')
else:
    try:
        gold_df = pd.read_csv(OUTPUT_PATH)
    except pd.errors.EmptyDataError:
        gold_df = pd.DataFrame()

    n_done = len(gold_df)
    print(f'Annotations done: {n_done}/75')

    if n_done == 0:
        print('No annotations yet — go back to the UI cell.')
    elif n_done < 75:
        print(f'Still {75 - n_done} candles to go. Run again after all 75 done.')
    else:
        kappa = compute_kappa(str(OUTPUT_PATH))
        if math.isnan(kappa):
            print('\nKappa undefined (all labels in one class or insufficient samples).')
        else:
            print(f'\nCohen kappa: {kappa:.3f}')
            if kappa < 0.6:
                print('GATE FAILED — kappa < 0.6. Investigate disagreements before training models.')
            else:
                print('Gate passed (kappa >= 0.6). Phase 4 can proceed.')

        if 'human_label' in gold_df.columns:
            disagree = gold_df[gold_df['programmatic_label'] != gold_df['human_label']]
            print(f'\nDisagreements: {len(disagree)}/{len(gold_df)} candles')
            if len(disagree) > 0:
                print(disagree[['candle_index', 'datetime', 'programmatic_label', 'human_label', 'annotator_note']].to_string(index=False))


Annotations done: 75/75

Cohen kappa: 1.000
Gate passed (kappa >= 0.6). Phase 4 can proceed.

Disagreements: 0/75 candles


## Criterion Ablation Analysis

The sparsity gate script (`scripts/count_valid_fvg.py`) generates a per-criterion ablation CSV.
This section loads it and visualises the rejection funnel — how many geometric FVGs survive each validity criterion.

Run `python scripts/count_valid_fvg.py` from the project root before running these cells.


In [6]:
# Criterion Ablation Analysis
# Requires: .nb-suite/analysis/criterion-ablation-<date>.csv (from scripts/count_valid_fvg.py)

from pathlib import Path
import pandas as pd
import plotly.graph_objects as go

analysis_dir = ROOT / '.nb-suite' / 'analysis'
ablation_files = sorted(analysis_dir.glob('criterion-ablation-*.csv'))

if not ablation_files:
    print("No ablation CSV found. Run: python scripts/count_valid_fvg.py")
else:
    latest = ablation_files[-1]
    print(f"Loading: {latest.name}")
    abl = pd.read_csv(latest, index_col=0)
    n_total = len(abl)

    # Per-criterion pass counts
    crit_labels = {
        'geometric_bull':    'Geometric (bull)',
        'geometric_bear':    'Geometric (bear)',
        'crit2_strict_bull': 'Crit2 Strict (bull)',
        'crit2_strict_bear': 'Crit2 Strict (bear)',
        'crit3_sr_bull':     'Crit3 S/R (bull)',
        'crit3_sr_bear':     'Crit3 S/R (bear)',
        'crit5_gann_bull':   'Crit5 Gann (bull)',
        'crit5_gann_bear':   'Crit5 Gann (bear)',
        'crit6_bos_bull':    'Crit6 BOS (bull)',
        'crit6_bos_bear':    'Crit6 BOS (bear)',
        'valid_bull':        'VALID BULL',
        'valid_bear':        'VALID BEAR',
    }
    cols = list(crit_labels.keys())
    counts = [int(abl[c].sum()) for c in cols]
    pcts   = [100.0 * c / n_total for c in counts]
    labels_str = [crit_labels[c] for c in cols]
    colors = ['#2196f3' if 'bull' in c else '#e53935' for c in cols]

    fig_bar = go.Figure(go.Bar(
        y=labels_str[::-1], x=counts[::-1],
        orientation='h',
        text=[f'{p:.1f}%' for p in pcts[::-1]],
        textposition='outside',
        marker_color=colors[::-1],
    ))
    fig_bar.update_layout(
        title=f'Per-Criterion Pass Counts — {n_total:,} total candles',
        xaxis_title='Number of bars (True)',
        height=520,
        margin=dict(l=160, r=80, t=60, b=40),
    )
    fig_bar.show()

    # Rejection funnel (bull path)
    n_geom = int(abl['geometric_bull'].sum())
    funnel_steps = [
        ('Geometric FVG', n_geom),
        ('+ Crit2 Strict', int((abl['geometric_bull'] & abl['crit2_strict_bull']).sum())),
        ('+ Crit3 S/R',    int((abl['geometric_bull'] & abl['crit2_strict_bull'] & abl['crit3_sr_bull']).sum())),
        ('+ Crit5 Gann',   int((abl['geometric_bull'] & abl['crit2_strict_bull'] & abl['crit3_sr_bull'] & abl['crit5_gann_bull']).sum())),
        ('+ Crit6 BOS',    int(abl['valid_bull'].sum())),
    ]
    funnel_names = [s[0] for s in funnel_steps]
    funnel_vals  = [s[1] for s in funnel_steps]

    fig_funnel = go.Figure(go.Funnel(
        y=funnel_names,
        x=funnel_vals,
        textinfo='value+percent initial',
        marker_color=['#1565c0', '#1976d2', '#1e88e5', '#42a5f5', '#00e676'],
    ))
    fig_funnel.update_layout(
        title='Bullish FVG Rejection Funnel (each criterion applied cumulatively)',
        height=420,
    )
    fig_funnel.show()


Loading: criterion-ablation-2026-05-10.csv


### Observation

The rejection funnel shows which criterion is the most aggressive filter. Based on the empirical results from the 14,686-candle SPY H1 dataset:

**Criterion #2 (strict reaction candle)** is the single most aggressive filter — only ~10% of geometric bullish FVGs have the reaction candle (N+2) closing inside the narrow gap zone. This is expected: SPY H1 gaps are tight (typically 0.5–1.5 pts), and a bar moving within the hour will more often overshoot than close precisely inside the zone.

**Criterion #5 (Gann position)** is the second most aggressive — only ~33% of FVGs surviving Crit2 have their bottom in the lower half of the 50-bar swing. This is structurally correct: a valid bullish FVG should form in a value area, not at extended highs.

**Criterion #6 (BOS recency)** filters aggressively from the remaining pool — only ~4% of FVGs surviving Crit5 have a confirming BOS within 20 bars. The 20-bar window is very selective on SPY H1, where structure breaks are common but the 20-bar recency window is strict.

The combined effect is a ~0.04% positive rate at default parameters — far below the 150-positive annotation threshold. The build log documents the full relaxation decision tree and the required user decision before re-annotation.
